**Programmer: python_scripts (Abhijith Warrier)**

🔐 PYTHON SCRIPT TO DEMO: ***HOW ADVERSARIAL INPUTS CAN FOOL AN ML MODEL*** 🐍

Explore how small, carefully introduced changes to input data can alter a machine learning model's predictions. This lab builds a baseline classifier, simulates adversarial input manipulation, observes the impact, and applies a simple defence.

---

## 🎯 Lab Objective

Machine learning models learn decision boundaries from their training data. An input that is close to one of these boundaries can sometimes be manipulated so that the model produces a different prediction.

In this lab, we'll train a simple classifier and deliberately modify selected input features to see whether we can change its prediction.

We'll then apply input validation as a basic defensive layer and verify its effect.

In [1]:
import numpy as np
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

---

## 🟢 Build the Baseline

We'll use the Iris dataset, which contains measurements for three different flower species.

First, we'll split the dataset into training and testing sets, standardise the features, and train a Logistic Regression classifier.

In [2]:
iris = load_iris()

X = pd.DataFrame(iris.data, columns=iris.feature_names)
y = pd.Series(iris.target)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = LogisticRegression(max_iter=1000)
model.fit(X_train_scaled, y_train)

baseline_predictions = model.predict(X_test_scaled)
baseline_accuracy = accuracy_score(y_test, baseline_predictions)

print(f"Baseline Accuracy: {baseline_accuracy:.2%}")

Baseline Accuracy: 92.11%


---

## 🔍 Select a Target Input

Rather than changing every test sample, we'll select one correctly classified sample.

We'll inspect its original feature values and record the model's prediction before introducing any manipulation.

In [3]:
correct_indices = np.where(baseline_predictions == y_test.to_numpy())[0]
target_index = correct_indices[0]

original_sample = X_test.iloc[target_index].copy()
original_scaled = scaler.transform(original_sample.to_frame().T)

original_prediction = model.predict(original_scaled)[0]
original_confidence = model.predict_proba(original_scaled).max()

print("Original Sample:")
print(original_sample)

print(f"\nActual Class: {iris.target_names[y_test.iloc[target_index]]}")
print(f"Predicted Class: {iris.target_names[original_prediction]}")
print(f"Confidence: {original_confidence:.2%}")

Original Sample:
sepal length (cm)    4.4
sepal width (cm)     3.2
petal length (cm)    1.3
petal width (cm)     0.2
Name: 42, dtype: float64

Actual Class: setosa
Predicted Class: setosa
Confidence: 98.46%


---

## 🔴 Simulate the Attack

Now we'll create controlled variations of the selected sample.

Instead of modifying the model or its training data, we'll slightly change the input features and repeatedly test the model until we find a modification that changes its prediction.

This demonstrates the basic idea behind an adversarial example: the model receives a modified input and behaves differently.

In [4]:
feature_ranges = X.max() - X.min()
perturbation_levels = np.linspace(0.01, 0.50, 50)

adversarial_sample = None
adversarial_prediction = original_prediction
used_perturbation = None

for level in perturbation_levels:
    candidate = original_sample.copy()

    candidate["petal length (cm)"] += feature_ranges["petal length (cm)"] * level
    candidate["petal width (cm)"] += feature_ranges["petal width (cm)"] * level

    candidate_scaled = scaler.transform(candidate.to_frame().T)
    candidate_prediction = model.predict(candidate_scaled)[0]

    if candidate_prediction != original_prediction:
        adversarial_sample = candidate
        adversarial_prediction = candidate_prediction
        used_perturbation = level
        break

---

## 🔍 Observe the Impact

Let's compare the original input with the modified version.

If the prediction changed, we can see exactly which features were altered and how the classifier responded.

In [5]:
if adversarial_sample is not None:
    comparison = pd.DataFrame({
        "Original": original_sample,
        "Adversarial": adversarial_sample,
        "Difference": adversarial_sample - original_sample
    })

    print(comparison)

    print(f"\nPerturbation Level: {used_perturbation:.2%}")
    print(f"Original Prediction: {iris.target_names[original_prediction]}")
    print(f"Adversarial Prediction: {iris.target_names[adversarial_prediction]}")
else:
    print("No prediction change was found within the tested perturbation range.")

No prediction change was found within the tested perturbation range.


---

## 🛡️ Apply the Defense

One simple defensive control is to validate whether incoming feature values remain within the range observed during training.

This is not a complete defence against adversarial machine learning, but it demonstrates an important principle: models should not blindly trust every input they receive.

In [6]:
training_min = X_train.min()
training_max = X_train.max()

def validate_input(sample):
    violations = []

    for feature in X.columns:
        if sample[feature] < training_min[feature] or sample[feature] > training_max[feature]:
            violations.append(feature)

    return violations

---

## 🧪 Verify the Defence

We'll now pass the adversarial sample through our validation layer before allowing it to reach the model.

If suspicious feature values fall outside the expected training range, the input can be flagged for additional handling instead of being trusted automatically.

In [7]:
if adversarial_sample is not None:
    violations = validate_input(adversarial_sample)

    if violations:
        print("⚠️ Suspicious input detected.")
        print(f"Features outside expected range: {violations}")
    else:
        print("✅ Input passed range validation.")
        print("Additional adversarial defenses may still be required.")
else:
    print("No adversarial sample available for validation.")

No adversarial sample available for validation.


---

## 📌 Security Takeaways

- Adversarial examples manipulate model inputs to influence predictions.
- Even correctly classified inputs can become unreliable after feature manipulation.
- Comparing baseline and manipulated predictions helps reveal model sensitivity.
- Input validation can detect some suspicious inputs, but it is not a complete adversarial defence.
- Robust AI systems require multiple defensive techniques rather than relying on the model alone.

---

## 🚀 Final Note

Adversarial machine learning shows that high model accuracy does not automatically mean a model is secure. Testing how models behave under manipulated inputs is an important part of building more robust AI systems.

---